# Scan Archive

In this notebook we are going through the following steps
- extract raw data from GE scan archive raw data file
- combine the raw data with the information from a pulseq seq file to create an ismrmrd file
- reconstruct the data

In [ ]:
import numpy as np
from einops import rearrange

from raw2ismrmrd.ismrmrd_from_sequence import ismrmrd_from_sequence

### Extract raw data from GE scan archive

To be able to get the raw data from a GE scan archive you need to have access to the `GERecon` toolbox
which requires a research agreement with GE. 

If you have got that then you can use the code provided in src/sa_to_npy.py 

Use the `read_archive` function to extract the raw data and save it as a numpy array.


### Combine raw data with pulseq information

In the next step we are going to combine the raw data from the scan archive file with the information about how the data
was acquired in the pulseq file. This only works if you used `labels` when creating the pulseq file to correctly identify 
at which k-space positions the data was acquired. Also other labels such as which echo number of which repetition number
the data is, is very helpful. Have a look here for an example of how this can be done: 
https://github.com/PTB-MR/mrseq/blob/main/src/mrseq/scripts/t1_t2_spiral_cmrf.py 

The raw data is passed to `ismrmrd_from_sequence` as a list of data objects of the shape `[coils readout]`. 
We use a list here because the different acquisitions do not have to have the same shape. 
E.g noise samples often have a different number of readout points compared to the data used for image reconstruction.

In [ ]:
kdata_raw = np.load('kspace.npy')
kdata_raw = rearrange(kdata_raw, 'readout acquisitions coils -> acquisitions coils readout')
kdata_raw_list = list(kdata_raw)
ismrmrd_from_sequence(kdata_raw_list, 'sequence.seq', 'kspace.mrd', replace_mrd=True)

### Image reconstruction

Now we can check if everything worked by reconstructed the image data. 
Here we use [MRpro](https://github.com/PTB-MR/MRpro) for the image reconstruction.

This data is a multi-echo spin echo acquisition where we can reconstruct images at different echo times. 
So we can also quickly fit a mono-exponential model to the data and get a T2 map.

In [ ]:
import matplotlib.pyplot as plt
import torch
from cmap import Colormap
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import CsmData
from mrpro.data import KData
from mrpro.data.traj_calculators import KTrajectoryCartesian
from mrpro.operators import DictionaryMatchOp
from mrpro.operators.models import MonoExponentialDecay

kdata = KData.from_file('kspace.mrd', KTrajectoryCartesian())
csm = CsmData.from_kdata_inati(kdata[0])
reco = DirectReconstruction(kdata=kdata, csm=csm)
idata = reco(kdata)

idat = idata.rss().abs().numpy().squeeze()
idat /= idat.max()
fig, ax = plt.subplots(2, idat.shape[0] // 2, figsize=(20, 10))
ax = ax.flatten()
for i in range(min(idat.shape[0], 40)):
    ax[i].imshow(idat[i, :, :], cmap='gray', vmin=0, vmax=0.9)
    ax[i].set_xticks([])
    ax[i].set_yticks([])

model = MonoExponentialDecay(decay_time=torch.as_tensor(idata.header.te))
dictionary = DictionaryMatchOp(model, 0).append(torch.ones(1), torch.linspace(0.01, 0.7, 200))
m0_match, t2_match = dictionary(idata.data.abs())

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for cax in ax.flatten():
    cax.set_xticks([])
    cax.set_yticks([])

im = ax[0].imshow(m0_match.squeeze().abs().numpy(), cmap='grey')
fig.colorbar(im, ax=ax[0], label='M0')

im = ax[1].imshow(t2_match.squeeze().numpy(), vmin=0, vmax=0.7, cmap=Colormap('navia').to_mpl())
fig.colorbar(im, ax=ax[1], label='T2 (s)')